# Customer Churn Analysis

This notebook walks through the full analysis — from loading and cleaning the data, to training a Decision Tree model, to interpreting the results.

**Goal:** Predict which telecom customers are likely to cancel their service so the business can step in early and retain them.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.data_processing import load_data, clean_data, encode_features, split_data
from src.model import train_model, evaluate_model

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 2. Load & Explore the Data

The dataset contains **7,043 customers** from a telecom company. Each row captures:
- **Who they are** — gender, senior citizen status, partner, dependents
- **What services they use** — phone, internet, streaming, security add-ons
- **Their account details** — tenure (how long they've been a customer), contract type, billing method, monthly and total charges
- **Whether they left** — the `Churn` column (Yes / No)

In [ ]:
df_raw = load_data()
print(f"Dataset shape: {df_raw.shape[0]} customers, {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

### Churn Breakdown

About **27%** of customers in this dataset churned. That imbalance is common in real-world churn data — most people stay, but the ones who leave are exactly who we need to identify.

In [ ]:
churn_counts = df_raw["Churn"].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Stayed", "Churned"], churn_counts.values,
       color=["#2ecc71", "#e74c3c"], edgecolor="black", linewidth=0.5)
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 30, str(v), ha="center", fontweight="bold")
ax.set_title("Customer Churn Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preparation

Before we can train a model, we need to:
1. **Fix data types** — `TotalCharges` has some blank values stored as text
2. **Drop irrelevant columns** — `customerID` is just a label, not a predictor
3. **Encode categories** — convert text values (like "Yes"/"No") into numbers the model can work with

In [ ]:
df_clean = clean_data(df_raw)
df_encoded = encode_features(df_clean)
print(f"Cleaned dataset: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")
df_encoded.head()

## 4. Exploratory Data Analysis

Let's look at how key variables differ between customers who stayed and those who left.

### Monthly Charges
Customers who left tend to have **higher monthly bills**. This could mean premium-tier customers feel they're not getting enough value.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = df_clean.copy()
plot_df["Churn"] = plot_df["Churn"].map({0: "Stayed", 1: "Churned"})
sns.boxplot(data=plot_df, x="Churn", y="MonthlyCharges", ax=ax,
            palette={"Stayed": "#2ecc71", "Churned": "#e74c3c"})
ax.set_title("Monthly Charges: Stayed vs Churned", fontsize=14, fontweight="bold")
ax.set_ylabel("Monthly Charges ($)")
plt.tight_layout()
plt.show()

### Tenure
Newer customers are far more likely to churn. Long-tenured customers have clearly found value and tend to stick around.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
stayed = df_clean[df_clean["Churn"] == 0]["tenure"]
churned = df_clean[df_clean["Churn"] == 1]["tenure"]
ax.hist(stayed, bins=30, alpha=0.6, label="Stayed", color="#2ecc71", edgecolor="black")
ax.hist(churned, bins=30, alpha=0.6, label="Churned", color="#e74c3c", edgecolor="black")
ax.set_title("Customer Tenure Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Number of Customers")
ax.legend()
plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
corr = df_encoded.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Train the Model

We use a **Decision Tree** classifier — a model that learns a series of yes/no questions about the data to make predictions. Think of it like a flowchart:

> *"Is the customer on a month-to-month contract?" → Yes → "Have they been a customer for less than 12 months?" → Yes → **High churn risk***

We set `max_depth=5` to keep the tree from memorising the training data (overfitting), and `class_weight='balanced'` to give extra attention to the smaller group of churned customers.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_encoded)
feature_names = list(X_train.columns)
print(f"Training set: {len(X_train)} samples")
print(f"Test set:     {len(X_test)} samples")

model = train_model(X_train, y_train, max_depth=5)
print("Model trained.")

## 6. Evaluate Performance

Key metrics explained:
- **Accuracy** — what percentage of all predictions were correct
- **Precision** — when the model says "this customer will churn", how often is it right
- **Recall** — out of all customers who actually churned, how many did the model catch
- **F1 Score** — the balance between precision and recall (higher is better)

In [ ]:
metrics = evaluate_model(model, X_test, y_test)
for name, value in metrics.items():
    print(f"  {name:>10}: {value:.2%}")

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))

### Confusion Matrix

This shows exactly where the model gets it right and where it makes mistakes. The ideal result is high numbers on the diagonal (correct predictions) and low numbers off the diagonal (errors).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. What Drives Churn?

The feature importance chart tells us which variables the model relies on most when deciding if a customer will leave.

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[-10:]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in indices], importances[indices],
        color="#3498db", edgecolor="black", linewidth=0.5)
ax.set_title("Top 10 Features Driving Churn", fontsize=14, fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.show()

### Decision Tree Visualization

Here's a look at the top levels of the tree — each box represents a decision point the model uses to classify customers.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(model, max_depth=3, feature_names=feature_names,
          class_names=["Stayed", "Churned"], filled=True, rounded=True,
          fontsize=9, ax=ax)
ax.set_title("Decision Tree (Top 3 Levels)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Key Takeaways

| Finding | Business Implication |
|---|---|
| **Contract type** is the strongest predictor | Month-to-month customers churn at much higher rates — incentivise longer contracts |
| **Tenure** matters a lot | New customers are most at risk — invest in strong onboarding experiences |
| **Monthly charges** correlate with churn | High-paying customers may feel under-served — proactive outreach for premium plans |
| **Internet service type** influences churn | Fibre optic customers churn more — may indicate service quality issues worth investigating |

### Next Steps
- Try ensemble models (Random Forest, Gradient Boosting) for potentially better accuracy
- Add cost-sensitive analysis — the cost of losing a customer vs. the cost of a retention offer
- Build a simple dashboard for the retention team to flag at-risk accounts